══════════════════════════════════════════════════════════════
 WEEK 9  |  DAY 6  |  PROJECT — FULL ML WORKFLOW

══════════════════════════════════════════════════════════════

 PROJECT GOAL
 ────────────
 Build a complete machine learning pipeline on the Titanic dataset:
 load and clean the data, engineer features, train a classifier
 inside a scikit-learn Pipeline, evaluate it, and save/reload the model.

 SKILLS USED
 ───────────
 - pandas: read_excel, fillna, drop, get_dummies
 - scikit-learn Pipeline with SimpleImputer and StandardScaler
 - train_test_split, RandomForestClassifier
 - accuracy_score, confusion_matrix, classification_report
 - joblib: dump and load for model persistence

 TIME:  ~45-60 minutes

 ─────────────────────────────────────────────────────────────
 ARCHITECTURE DECISION
 ─────────────────────
 Choosing between: batch scoring (scheduled job) vs real-time REST API vs streaming inference.
 Rule of thumb: use batch when predictions can be pre-computed on a schedule. Move to a REST API when users need sub-second responses. Use streaming only when predictions must react to live event streams.

══════════════════════════════════════════════════════════════

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

══════════════════════════════════════════════════════════════
 PART 1 — LOAD AND INSPECT THE DATA

══════════════════════════════════════════════════════════════
The Titanic dataset has one row per passenger.
The target column is "Survived" (0 = did not survive, 1 = survived).
We load it from Excel using pandas, then inspect shape and nulls.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
dataset_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "datasets", "titanic_train.xlsx")
df = pd.read_excel(dataset_path)

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nFirst 3 rows:")
print(df.head(3))

══════════════════════════════════════════════════════════════
 PART 2 — CLEAN AND PREPARE FEATURES

══════════════════════════════════════════════════════════════
We select a subset of columns that are useful for prediction.
Text columns like Name, Ticket, Cabin are dropped.
"Sex" is encoded as a binary integer (male=1, female=0).
"Embarked" gets one-hot encoding.
Missing Age values are left for the Pipeline to impute.

EXAMPLE ──────────────────────────────────────────────────────

Keep only these columns for the model

In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
target   = "Survived"

df_model = df[features + [target]].copy()

# Encode Sex as integer
df_model["Sex"] = df_model["Sex"].map({"male": 1, "female": 0})

# One-hot encode Embarked (drop_first avoids dummy variable trap)
df_model = pd.get_dummies(df_model, columns=["Embarked"], drop_first=True)

print("\nPrepared feature columns:")
print(df_model.columns.tolist())
print("\nNull counts after encoding:")
print(df_model.isnull().sum())

══════════════════════════════════════════════════════════════
 TASK 1 — SPLIT THE DATA

══════════════════════════════════════════════════════════════
Split df_model into training and test sets.
Use 80% for training and 20% for testing.
Set random_state=42 so results are reproducible.
The target column is "Survived".

Expected output (approximate, depends on dataset size):
  Training rows: 712
  Test rows:     179

--- starting data ---

In [ ]:
X = df_model.drop(columns=[target])
y = df_model[target]

In [ ]:
# X_train, X_test, y_train, y_test = ...

══════════════════════════════════════════════════════════════
 PART 3 — BUILD A PIPELINE AND TRAIN

══════════════════════════════════════════════════════════════
A scikit-learn Pipeline chains preprocessing steps with a model.
Step 1: SimpleImputer fills missing Age/Fare values with the median.
Step 2: StandardScaler normalizes numeric features to zero mean.
Step 3: RandomForestClassifier is the model.
Calling pipeline.fit() runs all steps in order on the training data.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("model",   RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Assumes Task 1 was completed and X_train / y_train exist
try:
    pipeline.fit(X_train, y_train)
    print("\nModel training complete.")
except NameError:
    print("\nSkipping fit — complete Task 1 first to define X_train and y_train.")

══════════════════════════════════════════════════════════════
 TASK 2 — EVALUATE THE MODEL

══════════════════════════════════════════════════════════════
Use the trained pipeline to predict on X_test.
Then print:
  - Accuracy score (rounded to 4 decimal places)
  - Confusion matrix
  - Full classification report

Expected output (approximate):
  Accuracy: 0.8212

  Confusion matrix:
  [[94 15]
   [17 53]]

  Classification report:
                precision    recall  f1-score   support
             0       0.85      0.86      0.85       109
             1       0.78      0.76      0.77        70

--- starting data ---
Use: pipeline, X_test, y_test




y_pred = ...

══════════════════════════════════════════════════════════════
 PART 4 — FEATURE IMPORTANCE

══════════════════════════════════════════════════════════════
Random Forest exposes feature importances — how much each
feature contributed to splitting decisions across all trees.
Higher = more important.  We access the fitted model via
pipeline.named_steps["model"].

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
try:
    rf_model      = pipeline.named_steps["model"]
    feature_names = X_train.columns.tolist()
    importances   = rf_model.feature_importances_

    importance_df = pd.DataFrame({
        "feature":    feature_names,
        "importance": importances,
    }).sort_values("importance", ascending=False)

    print("\nFeature importances:")
    print(importance_df.to_string(index=False))
except Exception:
    print("\nSkipping feature importance — complete Tasks 1 and 2 first.")

══════════════════════════════════════════════════════════════
 TASK 3 — SAVE AND RELOAD THE MODEL

══════════════════════════════════════════════════════════════
Save the trained pipeline to a file named "titanic_pipeline.pkl"
in the same folder as this script using joblib.dump().
Then reload it with joblib.load() and run one prediction on X_test
to confirm the reloaded model gives the same result as the original.

Expected output (approximate):
  Model saved to: ...titanic_pipeline.pkl
  Reloaded model accuracy: 0.8212
  Predictions match: True

--- starting data ---

In [ ]:
model_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "titanic_pipeline.pkl")

In [ ]:
# joblib.dump(...)
# reloaded = joblib.load(...)

══════════════════════════════════════════════════════════════
 PART 5 — PREDICT ON A NEW PASSENGER

══════════════════════════════════════════════════════════════
Once a model is saved, we can load it and predict survival for
a passenger we invent.  We must build a DataFrame with the same
column names as the training data.
Missing columns (like Embarked dummies) are filled with 0.

EXAMPLE ──────────────────────────────────────────────────────

A new passenger: 3rd class, male, age 22, travelling alone, paid 7.25

In [ ]:
new_passenger_data = {
    "Pclass": [3],
    "Sex":    [1],
    "Age":    [22.0],
    "SibSp":  [0],
    "Parch":  [0],
    "Fare":   [7.25],
}

try:
    # Add any dummy columns that training data had, defaulting to 0
    new_passenger = pd.DataFrame(new_passenger_data)
    for col in X_train.columns:
        if col not in new_passenger.columns:
            new_passenger[col] = 0
    new_passenger = new_passenger[X_train.columns]

    prediction = pipeline.predict(new_passenger)[0]
    probability = pipeline.predict_proba(new_passenger)[0]

    print("\nNew passenger prediction:")
    print(f"  Survived: {prediction}  (0 = No, 1 = Yes)")
    print(f"  Probability of survival: {probability[1]:.2%}")
except Exception:
    print("\nSkipping prediction — complete previous tasks first.")

print("\nProject complete.")

══════════════════════════════════════════════════════════════
 CONCEPT 4 — MODEL MONITORING: DETECTING DATA DRIFT
══════════════════════════════════════════════════════════════
 A model trained on historical data degrades when real-world data shifts.
 This is called data drift — the input distribution has changed since training.

 Types of drift:
   Data drift      : input feature distribution changes (e.g. new customer demographics)
   Concept drift   : the relationship between features and target changes
   Prediction drift: model output distribution shifts (outputs more 1s than expected)

 Simple detection — compare training statistics to production statistics:
   Mean shift   : |prod_mean - train_mean| > threshold
   Std shift    : |prod_std - train_std| > threshold × train_std
   Alert threshold: 2 standard deviations is a common starting point

 Production monitoring systems (Evidently, WhyLogs) do this at scale.
 We implement the core logic ourselves here.

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import numpy as np
import pandas as pd


def compute_stats(df: pd.DataFrame, columns: list) -> dict:
    """Compute mean and std for each column."""
    return {
        col: {"mean": df[col].mean(), "std": df[col].std()}
        for col in columns
    }


def detect_drift(train_stats: dict, prod_stats: dict,
                 mean_threshold: float = 0.2,
                 std_threshold: float = 0.5) -> list:
    """
    Compare production stats to training stats.
    Returns a list of drift alerts (empty = no drift detected).
    mean_threshold: fraction of train_mean that triggers alert (0.2 = 20%)
    std_threshold:  fraction of train_std shift that triggers alert
    """
    alerts = []
    for col in train_stats:
        t = train_stats[col]
        p = prod_stats.get(col, {})

        mean_shift = abs(p.get("mean", 0) - t["mean"])
        if t["mean"] != 0 and (mean_shift / abs(t["mean"])) > mean_threshold:
            alerts.append(
                f"DRIFT [{col}] mean: train={t['mean']:.2f} → prod={p['mean']:.2f} "
                f"(shift={mean_shift:.2f})"
            )

        std_shift = abs(p.get("std", 0) - t["std"])
        if t["std"] > 0 and (std_shift / t["std"]) > std_threshold:
            alerts.append(
                f"DRIFT [{col}] std: train={t['std']:.2f} → prod={p['std']:.2f} "
                f"(shift={std_shift:.2f})"
            )
    return alerts


# Training distribution — what the model was built on
np.random.seed(0)
train_df = pd.DataFrame({
    "age":    np.random.normal(35, 8, 1000),
    "income": np.random.normal(60000, 15000, 1000),
    "score":  np.random.normal(700, 50, 1000),
})

# Production distribution — 6 months later, customer base has shifted
prod_df = pd.DataFrame({
    "age":    np.random.normal(42, 10, 500),     # older customers
    "income": np.random.normal(62000, 14000, 500),
    "score":  np.random.normal(680, 70, 500),    # wider spread
})

cols = ["age", "income", "score"]
train_stats = compute_stats(train_df, cols)
prod_stats  = compute_stats(prod_df,  cols)

alerts = detect_drift(train_stats, prod_stats)
if alerts:
    print("Drift detected:")
    for a in alerts:
        print(" ", a)
else:
    print("No drift detected — distributions are stable.")

══════════════════════════════════════════════════════════════
 TASK 4 — BUILD A DRIFT MONITOR WITH ALERTING
══════════════════════════════════════════════════════════════
 Build a DriftMonitor class using the functions above.

   DriftMonitor:
     __init__(self, train_df, columns, mean_threshold=0.2, std_threshold=0.5)
       — computes and stores training stats on init

     check(self, prod_df) -> dict
       — computes prod stats, runs detect_drift, returns:
           {"alerts": [...], "drift_detected": bool, "features_checked": N}

     report(self, prod_df) -> None
       — calls check() and prints a formatted report

 Run the monitor on all three production batches in prod_batches below.
 Print the report for each.

 Expected output (values depend on random seed):
     === Drift Report: Batch 1 ===
     Features checked: 3
     Alerts: 0
     Status: STABLE

     === Drift Report: Batch 2 ===
     Features checked: 3
     Alerts: 2
     DRIFT [age] mean: train=35.XX → prod=50.XX (shift=...)
     DRIFT [score] std: train=50.XX → prod=90.XX (shift=...)
     Status: DRIFT DETECTED

 --- starting data ---

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(0)
train_df = pd.DataFrame({
    "age":    np.random.normal(35, 8, 1000),
    "income": np.random.normal(60000, 15000, 1000),
    "score":  np.random.normal(700, 50, 1000),
})

# Three production batches: stable, drifted, very drifted
prod_batches = [
    pd.DataFrame({   # Batch 1 — stable
        "age":    np.random.normal(35, 8, 300),
        "income": np.random.normal(60500, 15200, 300),
        "score":  np.random.normal(698, 52, 300),
    }),
    pd.DataFrame({   # Batch 2 — drifted
        "age":    np.random.normal(50, 8, 300),
        "income": np.random.normal(60000, 15000, 300),
        "score":  np.random.normal(700, 90, 300),
    }),
    pd.DataFrame({   # Batch 3 — severely drifted
        "age":    np.random.normal(25, 5, 300),
        "income": np.random.normal(40000, 8000, 300),
        "score":  np.random.normal(600, 100, 300),
    }),
]